# 레슨 10 - 프로덕션 환경의 AI 에이전트

이 수업에서는 Microsoft Agent Framework (Python)을 사용한 AI 에이전트를 위한 **프로덕션 패턴**을 배웁니다. 다룹니다:

- **관찰성** — 에이전트 상호작용에 타이밍과 로깅을 추가
- **평가** — 응답 품질을 점수화하기 위해 평가자 에이전트를 사용
- **비용 관리** — 토큰 최적화 및 모델 선택을 위한 전략

시나리오는 사용자가 여행을 계획하도록 돕는 **여행 에이전트**로, 그 위에 모니터링과 평가가 계층으로 추가되어 있습니다.


## 설정


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## 프로덕션 고려사항

AI 에이전트를 프로토타입에서 프로덕션으로 이동시키려면 세 가지 핵심 요소에 세심한 주의가 필요합니다:

1. **관찰성** — 에이전트가 무엇을 하는지, 얼마나 걸리는지, 어떤 도구를 호출하는지에 대한 가시성이 필요합니다. 추적 및 로깅 없이는 프로덕션 문제를 디버깅하는 것이 거의 불가능합니다.

2. **평가** — 자동화된 품질 검사는 시간이 지나도 에이전트의 응답이 정확하고 완전하며 유용하게 유지되는지 보장합니다. 평가자 에이전트는 정의된 기준에 따라 응답에 점수를 매길 수 있습니다.

3. **비용 관리** — 토큰 사용량은 비용에 직접적인 영향을 미칩니다. 프롬프트 최적화, 모델 선택, 캐싱과 같은 전략은 품질을 희생하지 않으면서 비용을 통제하는 데 도움이 됩니다.


## 관측 가능한 에이전트 구축

우리는 여행 도구들을 정의하고 에이전트 호출을 타이밍으로 감싸서 지연 시간을 모니터링할 수 있게 합니다. 운영 환경에서는 OpenTelemetry나 유사한 트레이싱 백엔드와 통합할 것입니다.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Evaluation Patterns

일반적인 프로덕션 패턴은 두 번째 에이전트를 **평가자**로 사용하는 것입니다. 평가자는 완전성, 정확성, 유용성과 같은 사전 정의된 기준에 따라 주요 에이전트의 응답을 점수화합니다.

이를 통해:
- 응답이 사용자에게 도달하기 전에 자동화된 품질 게이트
- 프롬프트나 모델이 변경될 때 회귀 감지
- 시간이 지남에 따른 에이전트 성능의 지속적인 모니터링


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## 비용 관리 전략

Controlling costs is critical for production AI agents. Here are key strategies:

| 전략 | 설명 |
|---|---|
| **프롬프트 최적화** | 시스템 지침을 간결하게 유지하세요. 입력 토큰을 줄이기 위해 중복된 컨텍스트를 제거하세요. |
| **모델 선택** | 분류 또는 추출과 같은 간단한 작업에는 더 작고 저렴한 모델(예: GPT-4o-mini)을 사용하고, 복잡한 추론에는 더 큰 모델을 예약하세요. |
| **캐싱** | 도구 결과와 자주 묻는 쿼리를 캐시하여 중복된 API 호출을 피하세요. |
| **토큰 예산** | `max_tokens` 제한을 설정하여 예상치 못하게 긴 응답을 방지하세요. |
| **배칭** | 가능한 경우 여러 사용자 쿼리를 하나의 API 호출로 그룹화하세요. |

실무에서는 계층화된 접근법이 잘 작동합니다: 간단한 요청은 빠르고 저렴한 모델로 라우팅하고 복잡한 쿼리만 더 능력 있는 모델로 에스컬레이션하세요.


## 요약

이번 레슨에서 다음을 배웠습니다:

1. **관찰 가능성 추가** 타이밍과 로깅을 사용해 에이전트 상호작용의 가시성을 높여 추적 및 모니터링의 기반을 마련했습니다.
2. **에이전트 응답 평가** 완전성, 정확성, 유용성을 점수화하는 평가자 에이전트를 사용해 응답을 자동으로 평가했습니다.
3. **비용 관리** 프롬프트 최적화, 모델 선택, 캐싱 및 토큰 예산을 통해 비용을 관리했습니다.

이러한 프로덕션 패턴은 AI 에이전트가 대규모 환경에서 신뢰할 수 있고 측정 가능하며 비용 효율적이도록 돕습니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 고지**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역에는 오류나 부정확성이 포함될 수 있음을 유의해 주십시오. 원문(원어)이 최종적이고 권위 있는 출처로 간주되어야 합니다. 중요한 정보의 경우 전문 번역가에 의한 인간 번역을 권장합니다. 본 번역의 사용으로 인해 발생한 오해나 오역에 대해서는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
